PySpark Guide



In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Practice") \
    .master("local[*]") \
    .getOrCreate()

In [ ]:
df = spark.read.format("csv").option('inferSchema', True).option('header', True).load('BigMartSales.csv')
df.show(5)

In [ ]:
df_json = spark.read.format("json").option('inferSchema', True).option('header', True).option('multiline', False).load('./drivers.json')
df_json.show(5)

In [ ]:
df.printSchema()
df_json.printSchema()

In [ ]:
from pyspark.sql.functions import collect_list

data1 = [
        ('1', 'Hrithik'),
        ('2', 'Neha'),
        ('3', 'Hardik')
        ]
schema1 = 'id STRING, name STRING'

df1 = spark.createDataFrame(data1, schema1);

data2 = [
        ('4', 'Hetvi'),
        ('5', 'Drashti'),
        ('6', 'Shaurya')
        ]
schema2 = 'id STRING, name STRING'

df2 = spark.createDataFrame(data2, schema2);
# df1.show()
# df2.show()

# df1.union(df2).show()
# df1.unionByName(df2).show()

data3 = [
        ('1', 'Hrithik'),
        ('2', 'Neha'),
        ('3', 'Hardik'),
        ('1', 'Hetvi'),
        ('2', 'Drashti'),
        ('3', 'Shaurya')
        ]
schema3 = 'id STRING, name STRING'

data3 = spark.createDataFrame(data3, schema3);

# Collect_List Function
data3.groupby('id').agg(collect_list('name')).show()

In [7]:
from pyspark.sql.functions import (
                                  col, lit, regexp_replace, StringType, initcap, lower, upper, 
                                  current_date, date_add, date_sub, datediff,date_format, split,
                                  explode, array_contains, sum, avg, collect_list, when, 
                                  row_number, rank, dense_rank, udf
                                  )

from pyspark.sql.window import Window

#df_sell = df.select(col('Item_Identifier'),col( 'Item_Type'), col('Item_MRP'))
#df.select(col('Item_Identifier').alias('Item_ID')).show()
#df.filter(col('Item_Fat_Content')=='Regular').show(5)
#df.filter((col('Item_Type')=='Soft Drinks') & (col('Item_Weight')<10)).show()
#df.filter((col('Outlet_Size').isNull()) & (col('Outlet_Location_Type').isin('Tier 1', 'Tier 2'))).show()
# df.withColumnRenamed('Item_Weight', 'Item_Wt').show(5)

# df = df.withColumn('Flag', lit('New'))
# df = df.withColumn('Multiply', col('Item_Weight') * col('Item_MRP'))

# df = df.withColumn('Item_Weight', col('Item_Weight').cast(StringType()))
# df.show(5)
# type(df.select('Item_Weight').take(1)[0][0])
# df.printSchema()

# df.select('Item_Weight').sort(col('Item_Weight').desc()).show(5)
# df.select('Item_Visibility').sort(col('Item_visibility').asc()).show()

# df.sort(
#     col("Item_Weight").desc(),
#     col("Item_Visibility").asc()
# ).show()

# df.limit(5).show()

# df.drop('Multiply').show(5)
# df.drop('Flag', 'Multiply').show(5)

# df.dropDuplicates(['Item_Identifier', 'Item_Type']).show()
# df.dropDuplicates(subset=['Item_Type']).show()
# df.distinct().show()

# String Functions
# df.select(initcap('Item_Type')).show(5)
# df.select(lower('Item_Type')).show(5)
# df.select(upper('Item_Type')).show(5)

# Date Functions
# df = df.withColumn('Curr_Date', current_date())
# df = df.withColumn('Week_After', date_add('Curr_Date', 7))
# df = df.withColumn('Week_Before', date_sub('Curr_Date', 7))
# # df.withColumn('Week_Before', date_add('Curr_Date', -7))
# df = df.withColumn('dateDiff', datediff('Week_After','Curr_Date'))
# df = df.withColumn('Curr_Date', date_format('Curr_Date', 'dd-MM-yyyy'))

# Handling NULL Values [Dropping OR Filling]
# df.dropna('all').show(20)
# df.dropna('any').show(20)
# df.dropna(subset=['Outlet_Size'])
# df.fillna('N/A', subset=['Outlet_Size']).show(20)

# Split and Indexing
# df.select('Outlet_Type').show(10)

df_exp = df.withColumn("Outlet_Type",split("Outlet_Type", " "))

# Explode Function
# df_exp = df_exp.withColumn("Outlet_Type",explode("Outlet_Type"))

# df_exp.show(20)

# Array_Contains Function
# df_exp.withColumn('Type1_Flag', array_contains('Outlet_Type', 'Type1')).show(10)

# GroupBy Function
# df_exp.groupBy('Item_Type', 'Outlet_Size').agg(
#   sum('Item_MRP').alias('Total_Sales'),
#   avg('Item_MRP').alias('Avg_Sales')
#   ).show()

# Pivot Function
# df_pivot = df_exp.select('Item_Type', 'Outlet_Size', 'Item_MRP')
# df_pivot.groupby('Item_Type').pivot('Outlet_Size').agg(avg('Item_MRP')).show()

# When-Otherwise Function (Like Case Statement in SQL)
# df_exp = df_exp.withColumn('Veg_Flag', when(col('Item_Type').isin('Meat', 'Frozen Foods'), 'Non-Veg').otherwise('Veg'))

# df_exp.withColumn('Test Flag', when(
#   (col('Veg_Flag')=='Veg') & (col('Item_MRP')<100), 'Veg & Cheap')\
#   .when((col('Veg_Flag')=='Veg') & (col('Item_MRP')>100), 'Veg & Expensive')\
#   .otherwise('Non-Veg')
#   ).show(20)


# Window Functions
# Row Number
# df_exp.withColumn(
#     "RowCol",
#     row_number().over(Window.orderBy("Item_Identifier"))
# ).show()

# Rank 
# df_exp.withColumn('Rank', rank().over(Window.orderBy(col('Item_Identifier').desc())))\
# .withColumn('Dense_Rank', dense_rank().over(Window.orderBy(col('Item_Identifier').desc()))).show(20)


# Dense Rank
# df_exp.withColumn('Rank', dense_rank().over(Window.orderBy('Item_Identifier'))).show()


# Cumulative Sum
# df_exp.withColumn('Cumm_Sum', sum('Item_MRP').over(Window.orderBy('Item_Type').rowsBetween(Window.unboundedPreceding, Window.currentRow))).show()


# df_exp.withColumn('Total_Sum', sum('Item_MRP').over(Window.orderBy('Item_Type').rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing))).show()


# User Defined Function [UDF]

def square(x):
    return x * x;

my_udf = udf(square)

# df_exp.withColumn('Multiplied MRPs', my_udf('Item_MRP')).show(10)


# Writing Data [Modes : append, overwrite, ignore, error]
# df.write.format('csv').save('./dataset/data')

# df.write.format('csv').mode('append').option('path', './dataset/data').save()

# df.write.format('csv').mode('overwrite').option('path', './dataset/data').save()

# df.write.format('csv').mode('error').option('path', './dataset/data').save()

# df.write.format('csv').mode('ignore').option('path', './dataset/data').save()


# Parquet Format
df.write.format('parquet').mode('ignore').option('path', './dataset/data').save()


# df.createTempView('My_view')

# df.select("Item_Fat_Content").distinct().show()
# df_sql = spark.sql("""
# SELECT *
# FROM My_view
# WHERE Item_Fat_Content = 'LF'
# """)
# df_sql.show()

# spark.catalog.listTables()

In [ ]:
# Joins [Inner, Left, Right, Full, Anti]
dataj1 = [('1','gaur','d01'),
          ('2','kit','d02'),
          ('3','sam','d03'),
          ('4','tim','d03'),
          ('5','aman','d05'),
          ('6','nad','d06')] 

schemaj1 = 'emp_id STRING, emp_name STRING, dept_id STRING' 

df1 = spark.createDataFrame(dataj1,schemaj1)

dataj2 = [('d01','HR'),
          ('d02','Marketing'),
          ('d03','Accounts'),
          ('d04','IT'),
          ('d05','Finance')]

schemaj2 = 'dept_id STRING, department STRING'

df2 = spark.createDataFrame(dataj2,schemaj2)

# df1.show()
# df2.show()

# Inner Join
# df1.join(df2, df1['dept_id'] == df2['dept_id'], 'inner').show()

# Left Join
# df1.join(df2, df1['dept_id'] == df2['dept_id'], 'left').show()

# Right Join
# df1.join(df2, df1['dept_id'] == df2['dept_id'], 'right').show()

# Full Join
# df1.join(df2, df1['dept_id'] == df2['dept_id'], 'full').show()

# Anti Join
# df1.join(df2, df1['dept_id'] == df2['dept_id'], 'anti').show()
# df2.join(df1, df1['dept_id'] == df2['dept_id'], 'anti').show()

df1.show()

In [ ]:
df = spark.read.parquet(
    "./dataset/data/part-00000-b9df5898-5424-4f25-92fb-a8e899162b41-c000.snappy.parquet"
)

df.printSchema()
df.show(5)